# Image Analyser — Training on Colab GPU

**Before running:**
1. Upload `training_dataset_v4_train_test.zip` to your Google Drive
2. Make sure your latest code is pushed to GitHub
3. Set Runtime → Change runtime type → **T4 GPU**

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/ai-property-triage'
BRANCH   = 'feature/image-analyser'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} https://github.com/Muhammadegb1/ai-property-triage.git {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -5

In [ ]:
!pip install -q torch torchvision pillow numpy

In [ ]:
import zipfile

# Path to the zip in Google Drive — update if you put it in a subfolder
ZIP_PATH = '/content/drive/MyDrive/training_dataset_v4_train_test.zip'
RAW_DIR  = '/content/ai-property-triage/services/image_analyser/data/raw'

os.makedirs(RAW_DIR, exist_ok=True)

print('Extracting dataset...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(RAW_DIR)

# Count extracted images
total = sum(len(files) for _, _, files in os.walk(RAW_DIR))
print(f'Extracted {total} files to {RAW_DIR}')

In [ ]:
import csv
from collections import Counter

LABELS_CSV = '/content/ai-property-triage/services/image_analyser/data/labels.csv'

with open(LABELS_CSV, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print(f'Total rows in labels.csv: {len(rows)}')

unscored = [r for r in rows if int(r['condition_score']) == 0]
if unscored:
    print(f'WARNING: {len(unscored)} rows have condition_score=0. Run auto_label.py first.')
else:
    print('All rows have condition scores assigned.')

splits = Counter(r['split'] for r in rows)
print(f'Splits: {dict(splits)}')

scores = Counter(r['condition_score'] for r in rows)
print('Condition score distribution:')
for s in sorted(scores):
    print(f'  Score {s}: {scores[s]}')

room_dist = Counter(r['room_type'] for r in rows)
print('\nRoom type distribution:')
for rt in sorted(room_dist):
    print(f'  {rt}: {room_dist[rt]}')

In [ ]:
%cd /content/ai-property-triage/services/image_analyser
!python train.py

In [ ]:
import shutil

CHECKPOINT_SRC = '/content/ai-property-triage/services/image_analyser/checkpoints/best_model.pth'
REPORT_SRC     = '/content/ai-property-triage/services/image_analyser/checkpoints/training_report.txt'
DRIVE_DEST     = '/content/drive/MyDrive/ai_property_triage_checkpoints/'

os.makedirs(DRIVE_DEST, exist_ok=True)

if os.path.exists(CHECKPOINT_SRC):
    shutil.copy(CHECKPOINT_SRC, DRIVE_DEST)
    print(f'Saved best_model.pth to {DRIVE_DEST}')
else:
    print('ERROR: best_model.pth not found — training may have failed')

if os.path.exists(REPORT_SRC):
    shutil.copy(REPORT_SRC, DRIVE_DEST)
    print(f'Saved training_report.txt to {DRIVE_DEST}')
    print('\n--- Training Report ---')
    with open(REPORT_SRC) as f:
        print(f.read())